# Auction Prediction Model - Regression & Classification
## Predicting Final Price and Win Probability

In [1]:
# Install required packages (uncomment if needed)
# !pip install xgboost scikit-learn pandas numpy joblib matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
    roc_curve
)

from xgboost import XGBClassifier, XGBRegressor
import joblib

print("Libraries loaded successfully!")

Libraries loaded successfully!


## 1. Load and Explore Data

In [2]:
# Load dataset
df = pd.read_csv("auction_dataset.csv")

print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nDataset Info:")
print(df.info())
print("\nBasic Statistics:")
print(df.describe())

Dataset Shape: (18000, 12)

First few rows:
   auction_id  starting_price  current_bid  bidders  total_bids  \
0           1             811         2338        9          42   
1           2            1906         3024       20         104   
2           3            1490         4820        2          36   
3           4            1237         3725        9          44   
4           5             396         1099       11          57   

   avg_increment  bid_velocity  auction_duration  time_remaining  final_price  \
0             82           8.0                58              70         2350   
1            193           8.0               180               3         6921   
2            151           8.0               192              51         4622   
3            125           8.0               115              41         3680   
4             40           8.0               110              76         1383   

   winner_id  will_win  
0         17         1  
1          5    

In [3]:
# Check for missing values
print("Missing Values:")
print(df.isnull().sum())
print("\nColumns:")
print(df.columns.tolist())

Missing Values:
auction_id          0
starting_price      0
current_bid         0
bidders             0
total_bids          0
avg_increment       0
bid_velocity        0
auction_duration    0
time_remaining      0
final_price         0
winner_id           0
will_win            0
dtype: int64

Columns:
['auction_id', 'starting_price', 'current_bid', 'bidders', 'total_bids', 'avg_increment', 'bid_velocity', 'auction_duration', 'time_remaining', 'final_price', 'winner_id', 'will_win']


## 2. Data Preparation & Feature Engineering

In [4]:
# Create a copy to avoid modifying original
df_processed = df.copy()

# Define target variables
# Regression target: final_price
y_regression = df_processed['final_price'].copy()

# Classification target: High price (above median) vs Low price
median_price = df_processed['final_price'].median()
df_processed['high_price'] = (df_processed['final_price'] > median_price).astype(int)
y_classification = df_processed['high_price'].copy()

print(f"Regression Target (final_price):")
print(f"  Min: {y_regression.min()}, Max: {y_regression.max()}, Mean: {y_regression.mean():.2f}")
print(f"\nClassification Target Distribution:")
print(y_classification.value_counts())
print(f"\nClass Balance: {y_classification.value_counts(normalize=True).round(4)}")

Regression Target (final_price):
  Min: 335, Max: 10787, Mean: 3412.26

Classification Target Distribution:
high_price
0    9002
1    8998
Name: count, dtype: int64

Class Balance: high_price
0    0.5001
1    0.4999
Name: proportion, dtype: float64


In [5]:
# Feature selection - IMPORTANT: Drop ID columns and target-related columns to avoid data leakage
# Exclude: auction_id (ID), final_price (target), winner_id (outcome), will_win (outcome), high_price (target)

feature_columns = [
    'starting_price',
    'current_bid',
    'bidders',
    'total_bids',
    'avg_increment',
    'bid_velocity',
    'auction_duration',
    'time_remaining'
]

X = df_processed[feature_columns].copy()

print(f"Features used: {feature_columns}")
print(f"\nFeature matrix shape: {X.shape}")
print(f"\nFeature Statistics:")
print(X.describe())

Features used: ['starting_price', 'current_bid', 'bidders', 'total_bids', 'avg_increment', 'bid_velocity', 'auction_duration', 'time_remaining']

Feature matrix shape: (18000, 8)

Feature Statistics:
       starting_price   current_bid       bidders    total_bids  \
count    18000.000000  18000.000000  18000.000000  18000.000000   
mean      1045.361389   2773.331500     11.020889     62.123111   
std        547.933836   1912.332572      5.431419     23.633025   
min        100.000000    100.000000      2.000000      6.000000   
25%        571.750000   1243.000000      6.000000     43.000000   
50%       1043.000000   2330.000000     11.000000     62.000000   
75%       1521.000000   3970.000000     16.000000     81.000000   
max       1999.000000   8893.000000     20.000000    118.000000   

       avg_increment  bid_velocity  auction_duration  time_remaining  
count   18000.000000  18000.000000      18000.000000    18000.000000  
mean      105.527167      7.990643        164.582778  

## 3. Standardize Features

In [6]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=feature_columns)

print("Features standardized successfully!")
print(f"Scaled features shape: {X_scaled.shape}")

Features standardized successfully!
Scaled features shape: (18000, 8)


## 4. Train-Test Split

In [7]:
# Split for regression
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_scaled, y_regression, test_size=0.2, random_state=42
)

# Split for classification
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_scaled, y_classification, test_size=0.2, random_state=42, stratify=y_classification
)

print("Train-Test Split Results:")
print(f"\nRegression:")
print(f"  Training set: {X_train_reg.shape[0]} samples")
print(f"  Test set: {X_test_reg.shape[0]} samples")
print(f"\nClassification:")
print(f"  Training set: {X_train_cls.shape[0]} samples")
print(f"  Test set: {X_test_cls.shape[0]} samples")
print(f"  Train class distribution: {dict(pd.Series(y_train_cls).value_counts())}")
print(f"  Test class distribution: {dict(pd.Series(y_test_cls).value_counts())}")

Train-Test Split Results:

Regression:
  Training set: 14400 samples
  Test set: 3600 samples

Classification:
  Training set: 14400 samples
  Test set: 3600 samples
  Train class distribution: {0: np.int64(7202), 1: np.int64(7198)}
  Test class distribution: {1: np.int64(1800), 0: np.int64(1800)}


## 5. Regression Model - Predicting Final Price

In [8]:
# Train regression model
reg_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='rmse',
    verbosity=0
)

print("Training Regression Model...")
reg_model.fit(X_train_reg, y_train_reg)
print("Regression model trained successfully!")

Training Regression Model...
Regression model trained successfully!


In [9]:
# Make predictions
y_pred_reg_train = reg_model.predict(X_train_reg)
y_pred_reg_test = reg_model.predict(X_test_reg)

# Calculate regression metrics
mae_train = mean_absolute_error(y_train_reg, y_pred_reg_train)
mae_test = mean_absolute_error(y_test_reg, y_pred_reg_test)

rmse_train = np.sqrt(mean_squared_error(y_train_reg, y_pred_reg_train))
rmse_test = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg_test))

r2_train = r2_score(y_train_reg, y_pred_reg_train)
r2_test = r2_score(y_test_reg, y_pred_reg_test)

print("="*60)
print("REGRESSION MODEL EVALUATION - Final Price Prediction")
print("="*60)
print(f"\nTraining Set Metrics:")
print(f"  MAE (Mean Absolute Error):   {mae_train:.2f}")
print(f"  RMSE (Root Mean Squared Error):  {rmse_train:.2f}")
print(f"  R² Score: {r2_train:.6f}")

print(f"\nTest Set Metrics:")
print(f"  MAE (Mean Absolute Error):   {mae_test:.2f}")
print(f"  RMSE (Root Mean Squared Error):  {rmse_test:.2f}")
print(f"  R² Score: {r2_test:.6f}")

print(f"\nOverfitting Analysis:")
print(f"  MAE Difference (Train-Test):  {abs(mae_train - mae_test):.2f}")
print(f"  R² Difference (Train-Test): {abs(r2_train - r2_test):.6f}")

REGRESSION MODEL EVALUATION - Final Price Prediction

Training Set Metrics:
  MAE (Mean Absolute Error):   112.31
  RMSE (Root Mean Squared Error):  155.33
  R² Score: 0.993453

Test Set Metrics:
  MAE (Mean Absolute Error):   143.75
  RMSE (Root Mean Squared Error):  208.30
  R² Score: 0.988294

Overfitting Analysis:
  MAE Difference (Train-Test):  31.44
  R² Difference (Train-Test): 0.005159


In [10]:
# Cross-validation for regression
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores_r2 = cross_val_score(reg_model, X_scaled, y_regression, cv=kfold, scoring='r2')
cv_scores_rmse = cross_val_score(reg_model, X_scaled, y_regression, cv=kfold, scoring='neg_mean_squared_error')
cv_scores_rmse = np.sqrt(-cv_scores_rmse)  # Convert to RMSE

print(f"\nCross-Validation Results (5-Fold):")
print(f"  R² Scores: {cv_scores_r2.round(6)}")
print(f"  Mean R² Score: {cv_scores_r2.mean():.6f} (+/- {cv_scores_r2.std():.6f})")
print(f"\n  RMSE Scores: {cv_scores_rmse.round(2)}")
print(f"  Mean RMSE:  {cv_scores_rmse.mean():.2f} (+/-  {cv_scores_rmse.std():.2f})")


Cross-Validation Results (5-Fold):
  R² Scores: [0.988199 0.988386 0.988456 0.987191 0.988233]
  Mean R² Score: 0.988093 (+/- 0.000461)

  RMSE Scores: [209.14 206.32 210.67 216.4  205.02]
  Mean RMSE:  209.51 (+/-  3.98)


## 6. Classification Model - Predicting High Price Category

In [11]:
# Train classification model
cls_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)

print("Training Classification Model...")
cls_model.fit(X_train_cls, y_train_cls)
print("Classification model trained successfully!")

Training Classification Model...
Classification model trained successfully!


In [12]:
# Make predictions
y_pred_cls_train = cls_model.predict(X_train_cls)
y_pred_cls_test = cls_model.predict(X_test_cls)
y_pred_proba_test = cls_model.predict_proba(X_test_cls)[:, 1]

# Calculate classification metrics
acc_train = accuracy_score(y_train_cls, y_pred_cls_train)
acc_test = accuracy_score(y_test_cls, y_pred_cls_test)
roc_auc = roc_auc_score(y_test_cls, y_pred_proba_test)

print("="*60)
print("CLASSIFICATION MODEL EVALUATION - High Price Prediction")
print("="*60)
print(f"\nAccuracy Metrics:")
print(f"  Training Accuracy: {acc_train:.6f}")
print(f"  Test Accuracy: {acc_test:.6f}")
print(f"  ROC-AUC Score: {roc_auc:.6f}")

print(f"\nDetailed Classification Report (Test Set):")
print(classification_report(y_test_cls, y_pred_cls_test, 
                          target_names=['Low Price', 'High Price']))

CLASSIFICATION MODEL EVALUATION - High Price Prediction

Accuracy Metrics:
  Training Accuracy: 0.996736
  Test Accuracy: 0.974167
  ROC-AUC Score: 0.997492

Detailed Classification Report (Test Set):
              precision    recall  f1-score   support

   Low Price       0.97      0.98      0.97      1800
  High Price       0.98      0.97      0.97      1800

    accuracy                           0.97      3600
   macro avg       0.97      0.97      0.97      3600
weighted avg       0.97      0.97      0.97      3600



In [13]:
# Confusion Matrix
cm = confusion_matrix(y_test_cls, y_pred_cls_test)
print("Confusion Matrix:")
print(cm)
print(f"\nTrue Negatives: {cm[0,0]}, False Positives: {cm[0,1]}")
print(f"False Negatives: {cm[1,0]}, True Positives: {cm[1,1]}")

Confusion Matrix:
[[1756   44]
 [  49 1751]]

True Negatives: 1756, False Positives: 44
False Negatives: 49, True Positives: 1751


In [14]:
# Cross-validation for classification
cv_scores_acc = cross_val_score(cls_model, X_scaled, y_classification, cv=kfold, scoring='accuracy')
cv_scores_auc = cross_val_score(cls_model, X_scaled, y_classification, cv=kfold, scoring='roc_auc')

print(f"\nCross-Validation Results (5-Fold):")
print(f"  Accuracy Scores: {cv_scores_acc.round(6)}")
print(f"  Mean Accuracy: {cv_scores_acc.mean():.6f} (+/- {cv_scores_acc.std():.6f})")
print(f"\n  ROC-AUC Scores: {cv_scores_auc.round(6)}")
print(f"  Mean ROC-AUC: {cv_scores_auc.mean():.6f} (+/- {cv_scores_auc.std():.6f})")


Cross-Validation Results (5-Fold):
  Accuracy Scores: [0.974167 0.973056 0.974167 0.974444 0.973056]
  Mean Accuracy: 0.973778 (+/- 0.000598)

  ROC-AUC Scores: [nan nan nan nan nan]
  Mean ROC-AUC: nan (+/- nan)


## 7. Feature Importance Analysis

In [15]:
# Feature importance for regression model
reg_importance = pd.DataFrame({
    'Feature': feature_columns,
    'Importance': reg_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("Feature Importance - Regression Model:")
print(reg_importance.to_string(index=False))

Feature Importance - Regression Model:
         Feature  Importance
     current_bid    0.609026
  starting_price    0.230579
      total_bids    0.090337
   avg_increment    0.041730
         bidders    0.023934
auction_duration    0.001876
  time_remaining    0.001472
    bid_velocity    0.001045


In [16]:
# Feature importance for classification model
cls_importance = pd.DataFrame({
    'Feature': feature_columns,
    'Importance': cls_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("Feature Importance - Classification Model:")
print(cls_importance.to_string(index=False))

Feature Importance - Classification Model:
         Feature  Importance
     current_bid    0.442875
  starting_price    0.224481
      total_bids    0.125284
   avg_increment    0.104957
         bidders    0.055708
    bid_velocity    0.016833
auction_duration    0.016258
  time_remaining    0.013604


## 8. Predictions on New Sample

In [17]:
# Make predictions on test sample
sample_index = 0
sample = X_test_cls.iloc[sample_index].values.reshape(1, -1)

# Regression prediction
predicted_final_price = reg_model.predict(sample)[0]

# Classification predictions
predicted_class = cls_model.predict(sample)[0]
predicted_probability_high = cls_model.predict_proba(sample)[0][1]
predicted_probability_low = cls_model.predict_proba(sample)[0][0]

print("="*60)
print("SAMPLE PREDICTION")
print("="*60)
print(f"\nRegression Model Output:")
print(f"  Predicted Final Price:  {predicted_final_price:.2f}")
print(f"  Actual Final Price (Test Set):  {y_test_reg.iloc[sample_index]:.2f}")
print(f"  Error:  {abs(predicted_final_price - y_test_reg.iloc[sample_index]):.2f}")

print(f"\nClassification Model Output:")
price_category = 'High Price' if predicted_class == 1 else 'Low Price'
print(f"  Predicted Category: {price_category}")
print(f"  Probability of High Price: {predicted_probability_high:.6f}")
print(f"  Probability of Low Price: {predicted_probability_low:.6f}")
print(f"  Actual Category (Test Set): {'High Price' if y_test_cls.iloc[sample_index] == 1 else 'Low Price'}")

SAMPLE PREDICTION

Regression Model Output:
  Predicted Final Price:  7672.06
  Actual Final Price (Test Set):  1883.00
  Error:  5789.06

Classification Model Output:
  Predicted Category: High Price
  Probability of High Price: 0.999981
  Probability of Low Price: 0.000019
  Actual Category (Test Set): High Price


## 9. Save Models

In [18]:
# Save trained models
joblib.dump(reg_model, 'price_pred.pkl')
joblib.dump(cls_model, 'classification.pkl')
joblib.dump(scaler, 'feature_scaler.pkl')

['feature_scaler.pkl']

## 10. Summary Report

In [19]:
print("\n" + "="*70)
print("FINAL MODEL SUMMARY REPORT")
print("="*70)

print("\n1. REGRESSION MODEL (Final Price Prediction)")
print("-" * 70)
print(f"   Test MAE:  {mae_test:.2f}")
print(f"   Test RMSE:  {rmse_test:.2f}")
print(f"   Test R² Score: {r2_test:.6f}")
print(f"   Mean CV R²: {cv_scores_r2.mean():.6f} (+/- {cv_scores_r2.std():.6f})")
print(f"   Interpretation: The model explains {r2_test*100:.2f}% of price variance")

print("\n2. CLASSIFICATION MODEL (High Price Category Prediction)")
print("-" * 70)
print(f"   Test Accuracy: {acc_test:.6f}")
print(f"   ROC-AUC Score: {roc_auc:.6f}")
print(f"   Mean CV Accuracy: {cv_scores_acc.mean():.6f} (+/- {cv_scores_acc.std():.6f})")
print(f"   Mean CV ROC-AUC: {cv_scores_auc.mean():.6f} (+/- {cv_scores_auc.std():.6f})")

print("\n3. MODEL COMPARISON")
print("-" * 70)
print(f"   Regression Overfitting: {'LOW' if abs(r2_train - r2_test) < 0.05 else 'MODERATE' if abs(r2_train - r2_test) < 0.1 else 'HIGH'}")
print(f"   Classification Overfitting: {'LOW' if abs(acc_train - acc_test) < 0.05 else 'MODERATE' if abs(acc_train - acc_test) < 0.1 else 'HIGH'}")

print("\n4. TOP 3 IMPORTANT FEATURES")
print("-" * 70)
print("   Regression Model:")
for idx, row in reg_importance.head(3).iterrows():
    print(f"     {idx+1}. {row['Feature']}: {row['Importance']:.6f}")
    
print("\n   Classification Model:")
for idx, row in cls_importance.head(3).iterrows():
    print(f"     {idx+1}. {row['Feature']}: {row['Importance']:.6f}")

print("\n" + "="*70)


FINAL MODEL SUMMARY REPORT

1. REGRESSION MODEL (Final Price Prediction)
----------------------------------------------------------------------
   Test MAE:  143.75
   Test RMSE:  208.30
   Test R² Score: 0.988294
   Mean CV R²: 0.988093 (+/- 0.000461)
   Interpretation: The model explains 98.83% of price variance

2. CLASSIFICATION MODEL (High Price Category Prediction)
----------------------------------------------------------------------
   Test Accuracy: 0.974167
   ROC-AUC Score: 0.997492
   Mean CV Accuracy: 0.973778 (+/- 0.000598)
   Mean CV ROC-AUC: nan (+/- nan)

3. MODEL COMPARISON
----------------------------------------------------------------------
   Regression Overfitting: LOW
   Classification Overfitting: LOW

4. TOP 3 IMPORTANT FEATURES
----------------------------------------------------------------------
   Regression Model:
     2. current_bid: 0.609026
     1. starting_price: 0.230579
     4. total_bids: 0.090337

   Classification Model:
     2. current_bid: 0.4